This code package requires Julia 1.10 or greater, CriticalTransitions 0.9.0.
In action.jl, in the function om_action, the lines:

    #if !sys.noise_type[:additive]
     #   throw(
      #      ArgumentError(
       #         "om_action currently implements the constant-diffusion Onsager-Machlup correction term and is only defined for additive noise. Use fw_action for the leading-order rate function under sta$
        #    ),
       # )
       
should be commented out and linked to the project

In [9]:
Base.active_project()
VERSION
Sys.BINDIR

"/Users/amethyst/.julia/juliaup/julia-1.10.11+0.aarch64.apple.darwin14/Julia-1.10.app/Contents/Resources/julia/bin"

In [1]:
using Pkg
Pkg.develop(path=expanduser("~/CriticalTransitions.jl"))

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`


In [2]:
using CriticalTransitions
println(pathof(CriticalTransitions))

Precompiling packages...
Info Given CriticalTransitions was explicitly requested, output will be shown live 
   4996.7 ms  ✓ CriticalTransitions
  1 dependency successfully precompiled in 6 seconds. 311 already precompiled.
  1 dependency had output during precompilation:
┌ CriticalTransitions
│  [Output was shown above]
└  


/Users/amethyst/CriticalTransitions.jl/src/CriticalTransitions.jl


In [3]:

import Pkg; 
Pkg.add("Polynomials")
Pkg.add("Random")
Pkg.add("Printf")
Pkg.add("DataFrames")
Pkg.add("CSV")
Pkg.add("HeuristicOptimizers")
Pkg.add("StaticArrays")
Pkg.add("LinearAlgebra")
Pkg.add("StaticArrays")
Pkg.add("CairoMakie")
Pkg.add("Plots")

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.to

In [4]:
# Stochastic Gottwald model
using CriticalTransitions
using Polynomials
using Random         
using Printf 
using HeuristicOptimizers
using StaticArrays, LinearAlgebra
using CairoMakie

In [5]:
using CSV, DataFrames
noisy = CSV.read("../data/cov_matrix.csv", DataFrame)

Row,Column1,t0,sigma_T,sigma_S,corr_TS,x_avg_corr,delta_avg_corr
,Int64,Float64,Float64,Float64,Float64,Float64,Float64
1,0,0.0,0.0161104,0.000659173,-0.995515,-0.000213723,8.53706e-5
2,1,0.01,0.0149483,0.000597823,-0.992881,0.0012944,-6.16318e-5
3,2,0.02,0.0169306,0.000687739,-0.997884,0.0039124,-0.000460277
4,3,0.03,0.0164058,0.000680231,-0.995801,0.00354039,-7.88602e-5
5,4,0.04,0.0181329,0.000745997,-0.997061,0.00446538,-7.01849e-5
6,5,0.05,0.0188292,0.000763559,-0.997688,0.00515267,8.04296e-6
7,6,0.06,0.0180236,0.000755107,-0.996198,0.005946,4.783e-5
8,7,0.07,0.0254057,0.00103727,-0.996984,0.00372712,0.000855425
9,8,0.08,0.016287,0.000674145,-0.997073,0.00866189,-0.000126408


In [153]:
# Smooth absolute value function
function smoothabs(x, xi=10000)
    x*tanh(x*xi)
end

#calculates fit of covariance matrix for each variable (Cov, sigma_T, sigma_S)

const t_data = collect(range(0.0, stop=100.0, length=nrow(noisy)))

function which_col(col)
    y = Float64.(noisy[!, col])
return y
end


coeff_sigmaT = fit(t_data,which_col("sigma_T"),3)
coeff_sigmaS = fit(t_data,which_col("sigma_S"),3)
coeff_corr   = fit(t_data,which_col("corr_TS"),3)

#store noise for plotting
temp_amplitude_log = Float64[]
temp_amplitude_shared_log = Float64[]
salt_amplitude_shared_log = Float64[]
salt_amplitude_log = Float64[]

function g(u, p, t)
    T = u[1]

    vT = coeff_sigmaT(T)   # fitted sigma_T value at T
    vS = coeff_sigmaS(T)   # fitted sigma_S value at T
    ρ  = coeff_corr(T)      # fitted correlation value at T

    #@assert vT > 0 "sigma_T(T) must be positive"
    #@assert vS > 0 "sigma_S(T) must be positive"
    #@assert abs(ρ) < 1 "corr_TS(T) must satisfy |ρ| < 1"
    σ = [vT                  0.0
    ρ * vS               vS * sqrt(1 - ρ^2)]

    #save time and matrix at each timestep
    push!(temp_amplitude_log, σ[1, 1])
    push!(temp_amplitude_shared_log, σ[1, 2])
    push!(salt_amplitude_shared_log, σ[2, 1])
    push!(salt_amplitude_log, σ[2, 2])

    return σ
end

function gottwald_noise(u, p, t)
    mu = 7.5
    epsilon_a = 0.34
    theta_0 = 1.0
    sigma_0 = 0.99

    x_avg_corr = 0.008654323346400778
    delta_avg_corr = 0.010372272583588681

    # Avoid mutating u; replace invalid values locally
    if any(isnan, u)
        T, S = 0.0, 0.0
    else
        T, S = u[1], u[2]
    end

    du1 = -1 / epsilon_a * (T - (theta_0 + x_avg_corr)) -
          T - mu * smoothabs(S - T) * T

    du2 = -S -
          mu * smoothabs(S - T) * S +
          sigma_0 + delta_avg_corr

    return SVector(du1, du2)
end

gottwald_noise (generic function with 1 method)

In [154]:
"""
A primer on large deviation theory
    
In the context of nonlinear dynamics, Large Deviation Theory provides tools to quantify the probability of rare events 
that deviate significantly from the system's typical behavior. These rare events might be extreme values of a system's output, 
sudden transitions between different states, or other phenomena that occur with very low probability but can have significant 
implications for the system's overall behavior.
Large deviation theory applies principles from probability theory and statistical mechanics to develop a 
rigorous mathematical description of these rare events. It uses the concept of a rate function, which measures the exponential 
decay rate of the probability of large deviations from the mean or typical behavior. This rate function plays a crucial role in 
quantifying the likelihood of rare events and understanding their impact on the system.
  
"""
#sigmas = [0.9, 0.95, 0.97, 0.99]
#for sigma in sigmas
attractor2 = [0.55, 0.55]   
attractor1 = [0.72,0.72]   #off state, Temperature, Salinity

#g is state dependent noise, based on the function written in the previous block
sys = CoupledSDEs(gottwald_noise,attractor1,attractor2,
    g=g,
    noise_strength = 1.0)

# `minimize_geometric_action` compute s the minimizer of the Freidlin-Wentzell action using the geometric minimum action 
#method (gMAM), to find the minimum action path (instanton) between an initial state x_i and final state x_f. 
#The Minimum Action Method (MAM) is a more traditional approach, while the Geometric Minimum Action Method (gMAM) is a 
#blend of the original MAM and the [string method](https://doi.org/10.1103/PhysRevB.66.052301).
gm = minimize_geometric_action(sys, attractor1,attractor2)
Ts = [p[1] for p in gm.path]
Ss = [p[2] for p in gm.path]

df = DataFrame(T = Ts, S = Ss)

CSV.write("instanton_path_back_99.csv", df)
#end

"instanton_path_back_99.csv"

In [37]:
path = gm.path
using Plots
Plots.gr() 
xs = [p[2] for p in gm.path]   # S coordinate
ys = [p[1] for p in gm.path]   # T coordinate

plt = Plots.plot(
    xs, ys;
    seriestype = :path,
    xlabel = "S",
    ylabel = "T",
    legend = false,
    aspect_ratio = :equal,
    xlims = (minimum(xs) - 0.01, maximum(xs) + 0.01),
    ylims = (minimum(ys) - 0.01, maximum(ys) + 0.01),
)

#display(plt)
Plots.savefig(plt, "instanton_path.png")

"/Users/amethyst/Documents/edge_tracking/code/instanton_path.png"

In [43]:
Ts = [p[1] for p in gm.path]
Ss = [p[2] for p in gm.path]

df = DataFrame(T = Ts, S = Ss)


using CSV
CSV.write("instanton_path.csv", df)

"instanton_path.csv"

In [104]:
#save noise dataframe
noise_df = DataFrame(
    temp_amplitude = temp_amplitude_log,
    temp_amplitude_shared = temp_amplitude_shared_log,
    salt_amplitude_shared = salt_amplitude_shared_log,
    salt_amplitude = salt_amplitude_log,
)

CSV.write("noise_matrix_log.csv", noise_df)

"noise_matrix_log.csv"